# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



The playbook converts the model's observed signals into a ranked list of content-review actions.

The highest-priority items are content pages where the model gives stronger evidence of a declining trend and where there are observable signals that a refresh may be useful, such as high impressions, low CTR, or older content.

The score is used for prioritization, not as a guarantee that a page needs a particular intervention.

### Reason codes

- `DECLINING_HIGH_VISIBILITY` — predicted decline with meaningful search visibility.
- `LOW_CTR_REFRESH` — low CTR combined with search visibility.
- `STALE_CONTENT` — content has been unchanged for a long period.
- `DECLINING_STALE` — predicted decline together with older content.
- `MONITOR` — weaker evidence; human review is recommended before action.

### Action labels

- `REFRESH_CONTENT`
- `REVIEW_TITLE_CTR`
- `REVIEW_STALENESS`
- `MONITOR`

In [ ]:
# ============================================================
# W07 - SECTION 1
# Ranked Actions + Reason Codes
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "acecod3z/Flyrankinternship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)


# ============================================================
# 2. TARGET AND FEATURES
# ============================================================

target = "trend_direction"

y = df[target]

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = df.drop(columns=drop_columns)

groups = df["client_id"]


# ============================================================
# 3. CLIENT-GROUPED SPLIT
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()


# ============================================================
# 4. PREPROCESSING
# ============================================================

numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            numeric_cols
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]),
            categorical_cols
        )
    ]
)


# ============================================================
# 5. RANDOM FOREST
# ============================================================

model = Pipeline([
    ("preprocessing", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

model.fit(
    X_train,
    y_train
)


# ============================================================
# 6. PREDICTIONS
# ============================================================

test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)

classes = model.named_steps["model"].classes_

probability_df = pd.DataFrame(
    test_probabilities,
    columns=classes,
    index=X_test.index
)

# Probability of "down"
if "down" in probability_df.columns:
    p_down = probability_df["down"]
else:
    p_down = pd.Series(
        0.0,
        index=X_test.index
    )


# ============================================================
# 7. BUILD ACTION QUEUE
# ============================================================

queue = df.loc[X_test.index].copy()

queue["p_down"] = p_down.values


# High visibility
queue["high_visibility"] = (
    queue["impressions_90d"].fillna(0) >=
    queue["impressions_90d"].fillna(0).median()
)

# Low CTR
queue["low_ctr"] = (
    queue["ctr"].fillna(queue["ctr"].median()) < 1.0
)

# Stale content
queue["stale"] = (
    queue["days_since_last_update"].fillna(0) >= 180
)


# ============================================================
# 8. TRANSPARENT ACTION SCORE
# ============================================================

queue["action_score"] = (
    queue["p_down"] * 100
    + queue["high_visibility"].astype(int) * 10
    + queue["low_ctr"].astype(int) * 5
    + queue["stale"].astype(int) * 5
)


# ============================================================
# 9. REASON CODE
# ============================================================

conditions = [
    (
        (queue["p_down"] >= 0.50)
        & queue["stale"]
    ),

    (
        (queue["p_down"] >= 0.50)
        & queue["high_visibility"]
    ),

    (
        queue["low_ctr"]
        & queue["high_visibility"]
    ),

    queue["stale"]
]

choices = [
    "DECLINING_STALE",
    "DECLINING_HIGH_VISIBILITY",
    "LOW_CTR_REFRESH",
    "STALE_CONTENT"
]

queue["reason_code"] = np.select(
    conditions,
    choices,
    default="MONITOR"
)


# ============================================================
# 10. ACTION LABEL
# ============================================================

action_conditions = [
    queue["reason_code"].isin([
        "DECLINING_STALE",
        "DECLINING_HIGH_VISIBILITY"
    ]),

    queue["reason_code"] == "LOW_CTR_REFRESH",

    queue["reason_code"] == "STALE_CONTENT"
]

action_choices = [
    "REFRESH_CONTENT",
    "REVIEW_TITLE_CTR",
    "REVIEW_STALENESS"
]

queue["action"] = np.select(
    action_conditions,
    action_choices,
    default="MONITOR"
)


# ============================================================
# 11. RANK
# ============================================================

queue = queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)


# ============================================================
# 12. DISPLAY TOP 20
# ============================================================

display(
    queue[
        [
            "rank",
            "content_id",
            "client_id",
            "action_score",
            "p_down",
            "reason_code",
            "action",
            "impressions_90d",
            "ctr",
            "days_since_last_update"
        ]
    ].head(20)
)

Dataset shape: (30000, 44)


,rank,content_id,client_id,action_score,p_down,reason_code,action,impressions_90d,ctr,days_since_last_update
0,1,content_9234f5075e7a,client_f369cb89fc,113.5,0.985,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,1305,0.38,20
1,2,content_9ac61c04930e,client_8527a891e2,113.0,0.980,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,1828,0.22,104
2,3,content_f6bf66378677,client_f369cb89fc,112.5,0.975,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,817,0.00,20
3,4,content_eb3b2c3bbc34,client_f369cb89fc,112.5,0.975,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,580,0.00,20
4,5,content_9e8671965fff,client_f369cb89fc,111.5,0.965,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,1322,0.08,20
5,6,content_ec257d803f69,client_4e07408562,111.5,0.965,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,963,0.10,104
6,7,content_d85f062b576f,client_4e07408562,110.5,0.955,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,2258,0.13,104
7,8,content_e5fd30b6e33b,client_f369cb89fc,110.5,0.955,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,687,0.29,20
8,9,content_8a12b4649424,client_f369cb89fc,110.0,0.950,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,645,0.47,20
9,10,content_a2d394e98056,client_4e07408562,110.0,0.950,DECLINING_HIGH_VISIBILITY,REFRESH_CONTENT,1220,0.00,104


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



### Intended use

The playbook is intended to help a human reviewer prioritize content for investigation.

It can support questions such as:

- Which pages should be reviewed first?
- Which pages show a combination of declining-trend probability and observable visibility?
- Which older pages may deserve a refresh review?
- Which pages may warrant a title/CTR review?

The ranked score is decision-support. It does not automatically establish that a page is declining, that a refresh will improve performance, or that a particular action will produce a business outcome.

### Limits

The model was evaluated on the available dataset and under a client-grouped validation design.

The client-grouped Random Forest result was:

- Accuracy: 72.40%
- Weighted F1: 69.35%

The validation result is an observed measurement on this dataset. It should not be interpreted as a guarantee of future performance or performance on other datasets.

The playbook is also not a production system. Actions require human review before implementation.

The observed relationships between freshness, visibility, CTR, and trend should be treated as directional decision-support signals rather than causal effects.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



Every ranked recommendation requires human review before an action is taken.

### Human review rules

A reviewer should check:

1. Whether the page is actually relevant to the business.
2. Whether the content is still factually correct.
3. Whether the page has important current information that should not be removed.
4. Whether the observed search and engagement signals are reliable.
5. Whether the recommended action makes sense for the page's intent.
6. Whether there are business, editorial, legal, or brand considerations not represented in the dataset.

### What should NOT be automated

The model should not automatically:

- publish rewritten content;
- change page titles or metadata;
- delete content;
- redirect URLs;
- change canonical URLs;
- change internal-link structures;
- make legal, medical, financial, or compliance decisions;
- declare that a content change caused a traffic improvement;
- treat a high model score as proof that a page must be refreshed.

The model can prioritize review, but the final content decision remains with a human.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The model should be monitored as a decision-support system rather than assumed to remain valid indefinitely.

### Monitoring signals

Useful monitoring checks include:

- distribution of model scores;
- distribution of predicted trend classes;
- proportion of pages receiving each action;
- changes in the distribution of key features such as impressions, CTR, position, and content age;
- measured precision of reviewed recommendations when human outcomes become available.

### Retrain triggers

A new model evaluation should be considered if:

- the distribution of important input features changes substantially;
- the distribution of predicted classes changes substantially;
- measured recommendation quality decreases;
- the content population changes materially;
- new historical labeled data becomes available;
- validation performance on a new time period or client group falls materially.

Retraining should not be triggered solely because a more complex model is available. A new model should demonstrate useful improvement against the existing baseline using an honest validation design.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


The ranked action queue is exported to `work/outputs/`.

The CSV is intentionally treated as a generated artifact rather than a committed data file. The notebook can regenerate it from the source data and model.

A small summary JSON is also exported as a reproducibility receipt for the paper.

In [ ]:
# ============================================================
# SECTION 5 — EXPORTS
# ============================================================

import os
import json


# Create output directory
os.makedirs(
    "work/outputs",
    exist_ok=True
)


# ============================================================
# 1. EXPORT RANKED QUEUE
# ============================================================

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "action_score",
    "p_down",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days"
]

output_path = (
    "work/outputs/"
    "w07_ranked_action_queue.csv"
)

queue[output_columns].to_csv(
    output_path,
    index=False
)


# ============================================================
# 2. SUMMARY RECEIPT
# ============================================================

summary = {
    "dataset_rows": int(len(df)),
    "test_rows": int(len(X_test)),
    "training_clients": int(
        df.iloc[train_idx]["client_id"].nunique()
    ),
    "testing_clients": int(
        df.iloc[test_idx]["client_id"].nunique()
    ),
    "random_seed": 42,
    "model": "RandomForestClassifier",
    "validation": "client-grouped",
    "queue_rows": int(len(queue))
}

json_path = (
    "work/outputs/"
    "w07_action_playbook_metrics.json"
)

with open(
    json_path,
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )


print("Queue written to:")
print(output_path)

print("\nMetrics receipt written to:")
print(json_path)

print("\nTop 10 actions:")

display(
    queue[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "action_score"
        ]
    ].head(10)
)

Queue written to:
work/outputs/w07_ranked_action_queue.csv

Metrics receipt written to:
work/outputs/w07_action_playbook_metrics.json

Top 10 actions:


,rank,content_id,action,reason_code,action_score
0,1,content_9234f5075e7a,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,113.5
1,2,content_9ac61c04930e,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,113.0
2,3,content_f6bf66378677,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,112.5
3,4,content_eb3b2c3bbc34,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,112.5
4,5,content_9e8671965fff,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,111.5
5,6,content_ec257d803f69,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,111.5
6,7,content_d85f062b576f,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,110.5
7,8,content_e5fd30b6e33b,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,110.5
8,9,content_8a12b4649424,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,110.0
9,10,content_a2d394e98056,REFRESH_CONTENT,DECLINING_HIGH_VISIBILITY,110.0


### Export check

The ranked queue was generated directly from the notebook.

The queue contains:

- a rank;
- an action score;
- a reason code;
- an action label;
- supporting observable signals.

The CSV is regenerated when the notebook is run and is not intended to be treated as a production decision system.

## 6. Self-check

- [x] Ranked actions are produced from the validated model output.
- [x] Every ranked item has an action and reason code.
- [x] Intended use is explicitly described as decision-support.
- [x] Model limitations are stated.
- [x] Human review is required before action.
- [x] No-go cases are explicitly listed.
- [x] Monitoring and retrain triggers are described.
- [x] The ranked queue is exported to `work/outputs/`.
- [x] The notebook uses a fixed random seed.
- [x] `trend_direction` and `trend_pct` are not used as model features.
- [x] Client IDs are used for grouping rather than prediction.
- [x] The plan is non-production and does not automatically change content.

In [ ]:
# ============================================================
# FINAL SELF-CHECK
# ============================================================

print("========== W07 SELF-CHECK ==========")

print(
    "Queue generated:",
    os.path.exists(
        "work/outputs/w07_ranked_action_queue.csv"
    )
)

print(
    "Metrics JSON generated:",
    os.path.exists(
        "work/outputs/w07_action_playbook_metrics.json"
    )
)

print(
    "Target excluded from features:",
    "trend_direction" not in X.columns
)

print(
    "Trend percentage excluded:",
    "trend_pct" not in X.columns
)

print(
    "Client ID excluded from features:",
    "client_id" not in X.columns
)

print(
    "Queue has action column:",
    "action" in queue.columns
)

print(
    "Queue has reason code:",
    "reason_code" in queue.columns
)

print(
    "Queue has ranking:",
    "rank" in queue.columns
)

print("\nW07 action playbook checks complete.")

========== W07 SELF-CHECK ==========
Queue generated: True
Metrics JSON generated: True
Target excluded from features: True
Trend percentage excluded: True
Client ID excluded from features: True
Queue has action column: True
Queue has reason code: True
Queue has ranking: True

W07 action playbook checks complete.


## 7. Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.